# Comparación estadística final de los diez mejores modelos

**experiment028 — Inferencia pareada de ROC AUC y exhaustividad (recall)**

Se recuperan las predicciones out-of-fold (OOF) de la corrida experiment024 documentada en experiment026, sin ajustar ningún modelo ni utilizar las etiquetas de prueba. Los diez finalistas se ordenan por ROC AUC, como en el notebook principal. Se comparan los 45 pares no ordenados.

**ROC AUC:** contraste bilateral pareado de DeLong como referencia principal para la métrica de Kaggle; bootstrap exacto estratificado y pareado como complemento. **Recall:** política de alerta con objetivo de recall 98 %, umbral calculado en los otros cuatro folds (misma implementación del notebook final). **Sensibilidad:** umbral fijo 0,50 para todos. Para recall, el contraste principal es McNemar exacto bilateral, con Holm sobre los 45 pares; Bonferroni simple y bootstrap centrado se reportan también. La sensibilidad constituye una familia exploratoria separada; no se combinan afirmaciones de las dos políticas sin un ajuste conjunto.

Los resultados son condicionales y exploratorios: los modelos fueron elegidos usando estas OOF, los folds de ajuste se solapan y las transacciones podrían ser dependientes. No son inferencia confirmatoria sobre modelos reentrenados ni sobre un conjunto externo.


In [ ]:
from __future__ import annotations
import os, zipfile, json, io, warnings, time, sys, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy, sklearn
from IPython.display import display, Markdown
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, log_loss, brier_score_loss

SEMILLA = 42
REANUDAR = True  # Recuperar bootstrap AUC compatible de experiment028 (no reentrenar).
CPU_JOBS = -1    # Configuración original; bootstrap usa hasta GRID_JOBS hilos.
GRID_JOBS = 4
DISPOSITIVO_XGB = 'auto'       # Configuración original; no se ajusta XGBoost.
DISPOSITIVO_CATBOOST = 'auto'  # Configuración original; no se ajusta CatBoost.
N_PARTICIONES = 5
N_BAGGING = 5
MUESTRA_GRAFICAS = 20_000  # Solo referencia de configuración: inferencia usa TODAS las filas.
RAIZ = Path.cwd()
if not (RAIZ / 'credit_card_fraud.ipynb').exists() and (RAIZ / 'final_project' / 'credit_card_fraud.ipynb').exists():
    RAIZ = RAIZ / 'final_project'
if not (RAIZ / 'experiments/experiment026/resultados/top10.csv').exists():
    RAIZ = next((p for p in [RAIZ,*RAIZ.parents] if (p/'experiments/experiment026/resultados/top10.csv').is_file()),RAIZ)
if not (RAIZ / 'experiments/experiment026/resultados/top10.csv').is_file():
    raise FileNotFoundError('Abrir Jupyter en la raíz de Credit-Fraud.')
DATOS_AGREGADOS = RAIZ / 'experiments' / 'experiment016' / 'resultados'
if not DATOS_AGREGADOS.is_dir() and (RAIZ / 'resultados').is_dir():
    DATOS_AGREGADOS = RAIZ / 'resultados'
INFORME_ORIGINAL = RAIZ / 'outputs' / 'informe_experiment024_es'
SALIDA = RAIZ / 'outputs' / 'experiment028'
FIGURAS = SALIDA / 'figuras'
FIGURAS.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette='dark', context='notebook')
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':150, 'axes.titlesize':12, 'axes.titleweight':'semibold'})
COLORES = sns.color_palette('dark')

def cerrar_figura(fig, nombre):
    fig.tight_layout()
    fig.savefig(FIGURAS / f'{nombre}.png', bbox_inches='tight', dpi=150)
    fig.savefig(FIGURAS / f'{nombre}.pdf', bbox_inches='tight')
    plt.show(); plt.close(fig)

DICCIONARIO = json.loads((DATOS_AGREGADOS/'diccionario_variables_es.json').read_text(encoding='utf-8')) if (DATOS_AGREGADOS/'diccionario_variables_es.json').exists() else {}
def etiqueta(col):
    return DICCIONARIO.get(col, str(col).replace('_',' ').capitalize())

print('Ubicación del proyecto:', RAIZ.resolve())
print('Inferencia experiment028 | Reanudar:', REANUDAR)
print('Versiones: Python', sys.version.split()[0], '| pandas', pd.__version__)

def anotaciones_matriz(m):
    total=np.sum(m)
    return np.array([[f'{int(valor):,}\n({valor/total:.2%})' for valor in fila] for fila in m],dtype=object)

ROOT=RAIZ; OUT=SALIDA
sys.path.insert(0,str(RAIZ/'experiments/experiment027'))
sys.path.insert(0,str(RAIZ/'experiments/experiment028'))
from recall_inference import read_labels,read_oof,decisions,pairwise_recall,p_matrix
from auc_inference import pairwise_auc
from joblib import cpu_count

CHECKPOINTS=RAIZ/'outputs/experiment024'
REFERENCE=RAIZ/'experiments/experiment026/resultados'
B=100_000
SEED=20261125  # Conservar semilla recall de experiment027 para comparabilidad.
B_AUC=2_000     # Bootstrap exacto de ambas clases; DeLong no depende de B.
ALPHA=.05
TARGET=.98
N_JOBS_AUC=max(1,min(GRID_JOBS,cpu_count() if CPU_JOBS==-1 else CPU_JOBS))


## 1. Recuperar los finalistas y verificar la corrida

Se leen exclusivamente las predicciones OOF guardadas por fold. Se comprueban la firma de la corrida, los índices de validación reconstruidos con semilla 42, la cobertura, el rango de probabilidades y las cuatro métricas publicadas (AUC, AP, log-loss y Brier). Se requiere el orden original del CSV de entrenamiento. La firma almacenada identifica la corrida; no se recalcula a partir de todos los datos y no prueba por sí sola la identidad de cada fila.


In [ ]:
ranking=pd.read_csv(REFERENCE/'top10.csv')
manifest=json.loads((REFERENCE/'resumen_final.json').read_text(encoding='utf8'))
y=read_labels(ROOT)
P,folds,audit=read_oof(CHECKPOINTS,ranking,manifest,y)
names=[f'M{i:02}' for i in ranking.puesto]
ranking.insert(0,'modelo',names)
display(ranking[['modelo','familia','roc_auc','average_precision']])
display(audit)
print(f'{len(y):,} transacciones | {int(y.sum()):,} fraudes | {len(names)} modelos')
ranking.to_csv(OUT/'modelos_top10.csv',index=False)
audit.to_csv(OUT/'auditoria_oof.csv',index=False)


## 2. Definición del estimando y políticas de alerta

Sobre los $n_+$ fraudes observados, $Z_{im}=\mathbf{1}\{\hat p_{im}\ge t_{im}\}$. La exhaustividad es $\hat R_m=n_+^{-1}\sum_{i:y_i=1} Z_{im}$. Para la política del 98 %, $t_{im}$ es el cuantil de las probabilidades de los positivos OOF de los otros folds (misma regla discreta y comparación >= que el informe principal). No se fija el recall de la propia muestra evaluada por construcción. Sin embargo, las OOF de calibración pueden proceder de modelos entrenados con las filas del fold de evaluación; no es calibración plenamente anidada. En la sensibilidad el umbral es constante 0,50.

Se reporta la tasa de falsos positivos (FPR) junto con recall; maximizar recall sin controlar las falsas alarmas no constituye una política operativa óptima.


In [ ]:
politicas={'recall98_cruzado':None,'umbral050':.5}
resultados={}
for policy,t in politicas.items():
    Z,thresholds=decisions(y,P,folds,target=TARGET,fixed_threshold=t)
    thresholds.to_csv(OUT/f'{policy}_umbrales.csv',index=False)
    models,pairs,n_patterns=pairwise_recall(y,Z,names,B=B,seed=SEED,alpha=ALPHA)
    models=models.merge(ranking[['modelo','familia','roc_auc']],on='modelo',validate='one_to_one')
    assert len(pairs)==45
    models.to_csv(OUT/f'{policy}_recall.csv',index=False)
    pairs.to_csv(OUT/f'{policy}_pares.csv',index=False)
    resultados[policy]=(models,pairs,n_patterns)
    print(policy,'| patrones conjuntos de detección:',n_patterns)
    display(models[['modelo','familia','VP','FN','FP','recall','recall_ci_low','recall_ci_high','tasa_falsas_alarmas']].round(6))

# Optional verification against the original local report, if available.
original=ROOT/'outputs/informe_experiment024_es/metricas_alerta_cruzadas.csv'
if original.is_file():
    t=pd.read_csv(original).set_index('Código')
    m=resultados['recall98_cruzado'][0].set_index('familia')
    for c1,c2 in [('FN','FN'),('FP','FP'),('recall','Recall cruzado')]:
        if not np.allclose(m[c1],t.loc[m.index,c2],rtol=0,atol=5e-10):
            raise ValueError(f'Política 98 % no coincide con informe original: {c1}')
    print('Política cruzada verificada contra el informe original.')


## 3. Bootstrap pareado, McNemar y multiplicidad

Para cada par (fila $A$, columna $B$), $D_i=Z_{iA}-Z_{iB}\in\{-1,0,1\}$ y $\hat\Delta=(b-c)/n_+$, donde $b$ y $c$ cuentan los fraudes detectados solo por A o solo por B, respectivamente.

**Bootstrap:** se remuestrean conjuntamente los vectores binarios de los diez modelos sobre positivos. Si un patrón aparece $n_k$ veces, sus frecuencias de remuestreo tienen distribución multinomial $\mathrm{Multinomial}(n_+;(n_k/n_+)_k)$. Esto es exactamente equivalente en distribución a remuestrear filas positivas con reemplazo y conserva el emparejamiento de todos los pares. Se usan 100 000 réplicas, IC percentiles puntuales del 95 % y p bilateral aproximado centrado en el nulo:
$$p_{boot}=\frac{1+\sum_{r=1}^{B}\mathbf{1}\{|\Delta_r^*-\hat\Delta|\ge|\hat\Delta|\}}{B+1}.$$
No se interpreta la proporción bootstrap de diferencias positivas como un p-valor. Los umbrales y modelos permanecen fijos durante el remuestreo; no se captura su incertidumbre de entrenamiento/calibración.

**McNemar exacto:** bajo $H_0:R_A=R_B$, y suponiendo pares independientes, $b\mid(b+c)\sim\mathrm{Binomial}(b+c,1/2)$. Se aplica el contraste bilateral exacto (si no hay discordancias, $p=1$). Es preferible al t-test pareado para datos discretos y pares con pocas discordancias.

**Correcciones:** para $m=45$, $p_i^{Bonf}=\min(1,mp_i)$. Ordenando $p_{(1)}\le \cdots\le p_{(m)}$, $p_{(k)}^{Holm}=\min(1,\max_{j\le k}(m-j+1)p_{(j)})$. Holm controla el error familiar bajo dependencia arbitraria si los p-valores individuales son válidos; el bootstrap centrado es aproximado. Los IC puntuales no son simultáneos. Referencias: Holm (1979), *Scandinavian Journal of Statistics*, 6, 65–70; Westfall, Troendle y Pennello (2010), [Multiple McNemar Tests](https://pmc.ncbi.nlm.nih.gov/articles/PMC2902578/).


In [ ]:
for policy,(models,pairs,_) in resultados.items():
    print(policy)
    display(pairs[['fila','columna','solo_fila','solo_columna','delta_pp','ci_low','ci_high',
                   'p_mcnemar','p_mcnemar_holm','p_mcnemar_bonferroni',
                   'p_bootstrap','p_bootstrap_holm','p_bootstrap_bonferroni']].sort_values('p_mcnemar_holm').style.format({
                       'delta_pp':'{:+.4f}','ci_low':'{:+.6f}','ci_high':'{:+.6f}',
                       **{c:'{:.3g}' for c in pairs if c.startswith('p_')}}))


## 4. Mapas de calor triangulares inferiores

Se utiliza seaborn.heatmap directamente sobre la matriz de p-valores originales, sin binarizar ni transformar los datos. Los tres paneles comparten escala lineal [0,1] y paleta secuencial rocket. Se anotan valores numéricos con cuatro cifras significativas; los CSV mantienen precisión completa. La diagonal y el triángulo superior se ocultan. Los asteriscos señalan p ≤ 0,05 en cada panel, según su corrección. Un p-valor pequeño no implica una magnitud importante, y uno grande no establece equivalencia.


In [ ]:
def plot_pvalues(pairs,policy,method):
    columns=[f'p_{method}',f'p_{method}_holm',f'p_{method}_bonferroni']
    titles=['Sin ajuste','Holm (45 pares)','Bonferroni (45 pares)']
    fig,axes=plt.subplots(1,3,figsize=(23,7.4),gridspec_kw={'wspace':.17})
    mask=np.triu(np.ones((len(names),len(names)),dtype=bool))
    for ax,col,title in zip(axes,columns,titles):
        M=p_matrix(pairs,names,col)
        M.to_csv(OUT/f'{policy}_{col}_matriz.csv')
        annotation=M.map(lambda p: '' if pd.isna(p) else f'{p:.4g}'+('*' if p<=ALPHA else ''))
        sns.heatmap(M,mask=mask,annot=annotation,fmt='',cmap='rocket',
            vmin=0,vmax=1,square=True,linewidths=.5,linecolor='white',
            cbar=False,ax=ax,annot_kws={'size':9})
        ax.set_title(title,fontweight='semibold')
        ax.set(xlabel='Modelo B (columna)',ylabel='Modelo A (fila)')
        ax.tick_params(axis='both',rotation=0)
    # Same continuous normalization for every panel; inset common colorbar.
    fig.subplots_adjust(right=.94,top=.87,bottom=.12)
    cax=fig.add_axes([.95,.23,.012,.49])
    fig.colorbar(axes[0].collections[0],cax=cax,label='p-valor original del panel')
    label={'mcnemar':'McNemar exacto','bootstrap':'Bootstrap pareado centrado','delong':'DeLong pareado'}[method]
    fig.suptitle(f'{policy} · {label}',fontsize=16,fontweight='semibold')
    fig.text(.5,.045,'* p ≤ 0,05 · valores originales, escala lineal común',ha='center')
    # Manual layout accommodates the shared colorbar without tight_layout moving it.
    fig.savefig(FIGURAS/f'{policy}_{method}_heatmap.png',dpi=150,bbox_inches='tight')
    fig.savefig(FIGURAS/f'{policy}_{method}_heatmap.pdf',bbox_inches='tight')
    plt.show();plt.close(fig)

for policy,(_,pairs,_) in resultados.items():
    for method in ('mcnemar','bootstrap'):
        plot_pvalues(pairs,policy,method)


## 5. Magnitud y dirección de las diferencias

Un valor positivo indica mayor recall del modelo de la fila respecto al de la columna, en puntos porcentuales. Los asteriscos identifican diferencias con p de McNemar-Holm ≤ 0,05. Las barras muestran intervalos bootstrap puntuales del 95 % frente a M01 (ganador nominal por AUC, no necesariamente por recall).


In [ ]:
for policy,(models,pairs,_) in resultados.items():
    M=p_matrix(pairs,names,'delta_pp')
    H=p_matrix(pairs,names,'p_mcnemar_holm')
    M.to_csv(OUT/f'{policy}_delta_pp_matriz.csv')
    lim=max(float(np.nanmax(np.abs(M.to_numpy()))),.0001)
    annotations=pd.DataFrame('',index=names,columns=names)
    for i in range(1,len(names)):
        for j in range(i):
            annotations.iloc[i,j]=f'{M.iloc[i,j]:+.3f}'+('*' if H.iloc[i,j]<=ALPHA else '')
    fig,ax=plt.subplots(figsize=(9,7))
    sns.heatmap(M,mask=np.triu(np.ones(M.shape,dtype=bool)),annot=annotations,fmt='',cmap='vlag',
                center=0,vmin=-lim,vmax=lim,square=True,ax=ax,cbar_kws={'label':'Diferencia de recall (pp)'})
    ax.set(title=f'{policy} · Δ recall (fila − columna)',xlabel='Modelo B',ylabel='Modelo A')
    cerrar_figura(fig,f'{policy}_delta_heatmap')

    q=pairs[pairs.columna==names[0]].copy()
    pos=np.arange(len(q))
    fig,ax=plt.subplots(figsize=(10,5.8))
    ax.hlines(pos,100*q.ci_low,100*q.ci_high,color=COLORES[0],lw=2)
    ax.scatter(q.delta_pp,pos,color=COLORES[0],zorder=3)
    ax.axvline(0,ls='--',color='gray')
    ax.set(yticks=pos,yticklabels=q.fila,xlabel='Diferencia frente a M01 (pp)',title=f'{policy} · IC bootstrap 95 % (puntuales)')
    ax.invert_yaxis()
    cerrar_figura(fig,f'{policy}_intervalos_vs_M01')


## 6. Comparación pareada de ROC AUC

Se utilizan las probabilidades OOF originales, sin umbrales, redondeo ni submuestreo. El AUC es un estadístico U de dos muestras:
$$\hat A_m=\frac{1}{n_+n_-}\sum_{i:y_i=1}\sum_{j:y_j=0}h(s_{im},s_{jm}),\quad h(a,b)=\mathbf1(a>b)+\tfrac12\mathbf1(a=b).$$

**DeLong (1988):** para cada modelo se calculan los promedios de $h$ por positivo y por negativo (componentes estructurales $V_+$ y $V_-$). La covarianza conjunta se estima mediante $\hat\Sigma=\operatorname{Cov}(V_+)/n_++\operatorname{Cov}(V_-)/n_-$. Para cada par, $\hat\Delta=\hat A_i-\hat A_j$, con varianza $\hat\Sigma_{ii}+\hat\Sigma_{jj}-2\hat\Sigma_{ij}$. El p bilateral usa una aproximación normal; se señala el caso degenerado de varianza cero. No se supone independencia entre clasificadores.

**Bootstrap:** cada réplica remuestrea todos los positivos y todos los negativos por separado, utilizando exactamente los mismos índices para los diez modelos. Se calcula el AUC ponderado exacto, incluyendo empates, mediante sumas acumuladas de multiplicidades de negativos ordenados. No es una aproximación lineal al bootstrap. Se usan 2 000 réplicas por defecto, con hasta cuatro hilos CPU y checkpoints cada 50 réplicas. DeLong no depende del número de réplicas y es el contraste principal; el p bootstrap centrado es aproximado, con resolución mínima $1/(B_{AUC}+1)$. Se ajusta Holm y Bonferroni sobre los 45 pares para cada método. IC percentiles puntuales del 95 %.

Referencia: [DeLong, DeLong y Clarke-Pearson, Biometrics 44(3), 837–845 (1988)](https://pubmed.ncbi.nlm.nih.gov/3203132/).


In [ ]:
cache_auc=OUT/'auc_bootstrap_PRIVATE.npz'
if not REANUDAR and cache_auc.exists():
    cache_auc.unlink()
inicio=time.perf_counter()
auc_models,auc_pairs,auc_cov,auc_draws=pairwise_auc(
    y,P,names,B=B_AUC,seed=SEMILLA,jobs=N_JOBS_AUC,alpha=ALPHA,
    checkpoint=cache_auc,source_signature=manifest['firma'],
    progress=lambda i,b: print(f'Bootstrap AUC: {i:,}/{b:,}',flush=True))
np.testing.assert_allclose(auc_models.roc_auc,ranking.roc_auc,rtol=0,atol=5e-10)
assert len(auc_pairs)==45
auc_models=auc_models.merge(ranking[['modelo','familia']],on='modelo',validate='one_to_one')
auc_models.to_csv(OUT/'roc_auc_modelos.csv',index=False)
auc_pairs.to_csv(OUT/'roc_auc_pares.csv',index=False)
auc_cov.to_csv(OUT/'roc_auc_cov_delong.csv')
display(auc_models.style.format({'roc_auc':'{:.9f}','auc_ci_low':'{:.9f}','auc_ci_high':'{:.9f}','se_delong':'{:.6g}'}))
display(auc_pairs.sort_values('p_delong_holm').style.format({
    'delta_auc':'{:+.9f}','delta_pp':'{:+.5f}','ci_low':'{:+.9f}','ci_high':'{:+.9f}',
    **{c:'{:.5g}' for c in auc_pairs if c.startswith('p_')}}))
print(f'Tiempo AUC (incluye caché si existe): {time.perf_counter()-inicio:.1f} s')


In [ ]:
for method in ('delong','bootstrap'):
    plot_pvalues(auc_pairs,'roc_auc',method)

M=p_matrix(auc_pairs,names,'delta_auc')
H=p_matrix(auc_pairs,names,'p_delong_holm')
M.to_csv(OUT/'roc_auc_delta_matriz.csv')
lim=max(float(np.nanmax(np.abs(M.to_numpy()))),1e-12)
annotation=pd.DataFrame('',index=names,columns=names)
for i in range(1,len(names)):
    for j in range(i):
        annotation.iloc[i,j]=f'{M.iloc[i,j]:+.6f}'+('*' if H.iloc[i,j]<=ALPHA else '')
fig,ax=plt.subplots(figsize=(10,8))
sns.heatmap(M,mask=np.triu(np.ones(M.shape,dtype=bool)),annot=annotation,fmt='',cmap='vlag',
            center=0,vmin=-lim,vmax=lim,square=True,ax=ax,cbar_kws={'label':'Δ ROC AUC'})
ax.set(title='Diferencia ROC AUC (fila − columna) · * Holm-DeLong ≤ 0,05',xlabel='Modelo B',ylabel='Modelo A')
cerrar_figura(fig,'roc_auc_delta_heatmap')

q=auc_pairs[auc_pairs.columna==names[0]].copy()
pos=np.arange(len(q))
fig,ax=plt.subplots(figsize=(10,5.8))
ax.hlines(pos,q.ci_low,q.ci_high,color=COLORES[0],lw=2)
ax.scatter(q.delta_auc,pos,color=COLORES[0],zorder=3)
ax.axvline(0,ls='--',color='gray')
ax.set(yticks=pos,yticklabels=q.fila,xlabel='Diferencia ROC AUC frente a M01',
       title='ROC AUC · IC bootstrap pareado 95 % (puntuales)')
ax.invert_yaxis()
cerrar_figura(fig,'roc_auc_intervalos_vs_M01')


### 6.1 Curvas ROC originales

Se muestran las curvas ROC de los diez modelos evaluadas sobre todas las predicciones OOF, con sus valores de AUC originales. La eliminación de puntos colineales de roc_curve no cambia la curva ni su área.


In [ ]:
fig,ax=plt.subplots(figsize=(10,7))
for j,name in enumerate(names):
    fpr,tpr,_=roc_curve(y,P[:,j],drop_intermediate=True)
    sns.lineplot(x=fpr,y=tpr,sort=False,estimator=None,errorbar=None,ax=ax,
                 color=COLORES[j%len(COLORES)],label=f'{name} · AUC {ranking.roc_auc.iloc[j]:.9f}',linewidth=1.5)
ax.plot([0,1],[0,1],ls='--',color='gray',lw=1)
ax.set(xlabel='Tasa de falsos positivos',ylabel='Tasa de verdaderos positivos',title='Curvas ROC OOF · diez finalistas',xlim=(0,1),ylim=(0,1.01))
ax.legend(loc='lower right',fontsize=9)
cerrar_figura(fig,'roc_auc_curvas_top10')


## 7. Resumen y límites de interpretación

Se comparan por separado igualdad de ROC AUC e igualdad de recall; la selección original se mantiene por AUC. Holm controla multiplicidad por familia de 45 pares (métrica/política/método) bajo los supuestos de cada test; no controla conjuntamente todas las familias. No corrige el sesgo de selección de los 40 candidatos. La sensibilidad 0,50 se considera descriptiva por separado. Las diferencias de recall deben interpretarse junto con FPR y magnitud. No se concluye equivalencia a partir de p > 0,05; para ello se necesitaría un margen de equivalencia preespecificado. Se guardan únicamente resultados agregados; las predicciones originales y los datos de Kaggle permanecen privados.


In [ ]:
summary=[]
for policy,(models,pairs,n_patterns) in resultados.items():
    winner=models.sort_values(['recall','tasa_falsas_alarmas'],ascending=[False,True]).iloc[0]
    summary.append(dict(politica=policy,n_positivos=int(y.sum()),n_pares=len(pairs),n_patrones=n_patterns,
        mayor_recall=winner.modelo,recall=float(winner.recall),fpr=float(winner.tasa_falsas_alarmas),
        mcnemar_sin_ajuste=int((pairs.p_mcnemar<=ALPHA).sum()),
        mcnemar_holm=int((pairs.p_mcnemar_holm<=ALPHA).sum()),
        mcnemar_bonferroni=int((pairs.p_mcnemar_bonferroni<=ALPHA).sum()),
        bootstrap_holm=int((pairs.p_bootstrap_holm<=ALPHA).sum())))
summary=pd.DataFrame(summary)
display(summary)
summary.to_csv(OUT/'resumen_inferencia.csv',index=False)
auc_summary=pd.DataFrame([dict(metrica='ROC AUC',n_pares=len(auc_pairs),delong_sin_ajuste=int((auc_pairs.p_delong<=ALPHA).sum()),delong_holm=int((auc_pairs.p_delong_holm<=ALPHA).sum()),delong_bonferroni=int((auc_pairs.p_delong_bonferroni<=ALPHA).sum()),bootstrap_holm=int((auc_pairs.p_bootstrap_holm<=ALPHA).sum()))])
display(auc_summary)
auc_summary.to_csv(OUT/'resumen_auc.csv',index=False)
meta=dict(experiment='experiment028',source='experiment024',source_signature=manifest['firma'],
    n_models=len(names),n_rows=len(y),n_positive=int(y.sum()),B=B,seed=SEED,alpha=ALPHA,target_recall=TARGET,
    primary_policy='recall98_cruzado',primary_test='exact two-sided McNemar, Holm over 45 pairs',
    auc_primary_test='paired DeLong, Holm over 45 pairs',auc_bootstrap_B=B_AUC,auc_seed=SEMILLA,auc_jobs=N_JOBS_AUC,
    bootstrap='joint binary pattern multinomial, percentile CI, centered p-value',
    versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,scipy=scipy.__version__,sklearn=sklearn.__version__),
    scope='Exploratory conditional on selected OOF models and fixed thresholds; no training uncertainty.')
(OUT/'manifest.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf8')
print('Resultados agregados:',OUT)
